# 69 - Cross-Dataset Late Fusion TL → Primer Test (Skema 2 Extension)

Melengkapi Skema 2 dengan **Late Fusion TL** yang terlewat di nb 63 (hanya 6 model) dan nb 68 (Early Fusion). **Inference-only** — reuse checkpoint `CNN_TL_B1` + `FCNN_B1` dari nb 65 yang dijalankan Skema 1 per source.

**Strategi:**
1. Load `CNN_TL_B1` + `FCNN_B1` yang di-train di source dataset
2. Inference keduanya di Primer test → 2 softmax distributions
3. Weighted averaging: `score = w * cnn_softmax + (1-w) * fcnn_softmax`
4. Grid search `w` dengan **2 strategi** untuk fair comparison:
   - **A (Primer val)**: tune `w` di Primer val (579 imgs), eval di Primer test
   - **B (Source val)**: tune `w` di source dataset val (pure zero-shot target-agnostic)
5. Laporkan keduanya — let user/dosen decide mana yang lebih fair untuk paper/tesis

**Matriks:** 4 source (CK+/JAFFE/RAF-DB/KDEF) × 2 class (7c/4c) × 2 strategy = **16 config pairs** (tapi 1 weight opt per pair = 8 total Late Fusion TL evaluations × 2 strategi)

**Output:**
- Update `models/benchmark/crossdataset/cross_{source}_{num}c.json` dengan 2 keys:
  - `Late_Fusion_TL_B1` → strategi A (Primer val tune)
  - `Late_Fusion_TL_B1_srcval` → strategi B (source val tune)
- Update `all_cross_results.json` combined

**Prerequisites:**
1. Checkpoint `CNN_TL_B1` + `FCNN_B1` ada di `models/benchmark/{ds}/...` ✓ (dari nb 65)
2. Primer conf60 val + test set ada

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer, EmotionFCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 64

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
MODELS_DIR = PROJECT_ROOT / 'models' / 'benchmark'
CROSS_DIR = MODELS_DIR / 'crossdataset'
CROSS_DIR.mkdir(parents=True, exist_ok=True)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

In [ ]:
# ── Load Primer val + test ──

primer_v_img = np.load(PRIMER_DIR / 'X_val_images.npy')
primer_v_lm  = np.load(PRIMER_DIR / 'X_val_landmarks.npy')
primer_v_y7  = np.load(PRIMER_DIR / 'y_val.npy')
primer_t_img = np.load(PRIMER_DIR / 'X_test_images.npy')
primer_t_lm  = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
primer_t_y7  = np.load(PRIMER_DIR / 'y_test.npy')

primer_v_y4 = REMAP_4[primer_v_y7]
primer_t_y4 = REMAP_4[primer_t_y7]

print(f'Primer val:  img={primer_v_img.shape}  lm={primer_v_lm.shape}  y={primer_v_y7.shape}')
print(f'Primer test: img={primer_t_img.shape}  lm={primer_t_lm.shape}  y={primer_t_y7.shape}')

In [ ]:
# ── Helpers (reuse convention dari nb 65/68) ──

def _subject_split(subjects, seed=42, train_ratio=0.8, val_ratio=0.1):
    rng = np.random.RandomState(seed)
    uniq = np.array(sorted(set(subjects.tolist())))
    rng.shuffle(uniq)
    n = len(uniq)
    n_tr = int(n * train_ratio); n_v = int(n * val_ratio)
    return set(uniq[:n_tr].tolist()), set(uniq[n_tr:n_tr+n_v].tolist()), set(uniq[n_tr+n_v:].tolist())


def load_source_val(dataset_name, num_classes):
    """Return source val set (img, lm, y) — untuk strategi B grid search."""
    if dataset_name == 'rafdb':
        d = BENCHMARK_DIR / f'rafdb_{num_classes}class'
        X = np.load(d / 'X_train_images.npy')
        L = np.load(d / 'X_train_landmarks.npy')
        y = np.load(d / 'y_train.npy')
        idx_tr, idx_v = train_test_split(np.arange(len(y)), test_size=0.1, stratify=y, random_state=42)
        return X[idx_v], L[idx_v], y[idx_v]

    if dataset_name == 'kdef':
        d = BENCHMARK_DIR / f'kdef_{num_classes}class'
        return (np.load(d / 'X_val_images.npy'),
                np.load(d / 'X_val_landmarks.npy'),
                np.load(d / 'y_val.npy'))

    if dataset_name in ('ckplus', 'jaffe'):
        if dataset_name == 'ckplus' and num_classes == 4:
            d = BENCHMARK_DIR / 'ckplus_4class_contempt'
        else:
            d = BENCHMARK_DIR / f'{dataset_name}_{num_classes}class'
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subjects = np.load(d / 'subjects.npy', allow_pickle=True)
        _, v_subs, _ = _subject_split(subjects)
        v_idx = np.where(np.isin(subjects, list(v_subs)))[0]
        return X[v_idx], L[v_idx], y[v_idx]

    raise ValueError(f'Unknown dataset {dataset_name}')


def checkpoint_dir(dataset_name, num_classes, model_key):
    """Match nb 65/66 convention."""
    if dataset_name in ('ckplus', 'jaffe'):
        return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c' / model_key
    return MODELS_DIR / dataset_name / f'{num_classes}c' / model_key


def make_cnn_loader(img, y, batch_size=BATCH_SIZE):
    t = torch.from_numpy(img).permute(0, 3, 1, 2).contiguous()
    ys = torch.from_numpy(y).long()
    return DataLoader(TensorDataset(t, ys), batch_size=batch_size, shuffle=False,
                      num_workers=0, pin_memory=True)


def make_fcnn_loader(lm, y, batch_size=BATCH_SIZE):
    t = torch.from_numpy(lm)
    ys = torch.from_numpy(y).long()
    return DataLoader(TensorDataset(t, ys), batch_size=batch_size, shuffle=False,
                      num_workers=0, pin_memory=True)


@torch.no_grad()
def batched_softmax(model, loader):
    model.eval()
    probs = []
    for xb, _ in loader:
        xb = xb.to(device)
        probs.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


def metrics_triple(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }


def grid_search_w(cnn_probs, fcnn_probs, y_true):
    """Find best w ∈ [0, 1] by macro F1."""
    best_f1, best_w = 0.0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * cnn_probs + (1 - w) * fcnn_probs).argmax(axis=1)
        f = f1_score(y_true, pr, average='macro', zero_division=0)
        if f > best_f1:
            best_f1, best_w = f, w
    return float(best_w), float(best_f1)


def cross_late_fusion_tl(dataset_name, num_classes):
    print(f"\n{'='*70}")
    print(f'  Cross: {dataset_name.upper()} {num_classes}c -> Primer test (Late Fusion TL)')
    print(f"{'='*70}")

    # Select Primer labels matching num_classes
    y_primer_val = primer_v_y7 if num_classes == 7 else primer_v_y4
    y_primer_test = primer_t_y7 if num_classes == 7 else primer_t_y4

    # 1. Load checkpoints
    cnn_ckpt = checkpoint_dir(dataset_name, num_classes, 'CNN_TL_B1') / 'model.pth'
    fcnn_ckpt = checkpoint_dir(dataset_name, num_classes, 'FCNN_B1') / 'model.pth'
    if not cnn_ckpt.exists() or not fcnn_ckpt.exists():
        print(f'  [SKIP] missing checkpoint(s): cnn={cnn_ckpt.exists()}  fcnn={fcnn_ckpt.exists()}')
        return None

    cnn_tl = EmotionCNNTransfer(num_classes=num_classes).to(device)
    cnn_tl.load_state_dict(torch.load(cnn_ckpt, map_location=device, weights_only=True))
    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    fcnn.load_state_dict(torch.load(fcnn_ckpt, map_location=device, weights_only=True))

    # 2. Inference on Primer val + test
    pv_cnn_loader = make_cnn_loader(primer_v_img, y_primer_val)
    pv_fcnn_loader = make_fcnn_loader(primer_v_lm, y_primer_val)
    pt_cnn_loader = make_cnn_loader(primer_t_img, y_primer_test)
    pt_fcnn_loader = make_fcnn_loader(primer_t_lm, y_primer_test)

    pv_cnn = batched_softmax(cnn_tl, pv_cnn_loader)
    pv_fcnn = batched_softmax(fcnn, pv_fcnn_loader)
    pt_cnn = batched_softmax(cnn_tl, pt_cnn_loader)
    pt_fcnn = batched_softmax(fcnn, pt_fcnn_loader)

    # ── Strategi A: tune w on Primer val ──
    w_a, val_f1_a = grid_search_w(pv_cnn, pv_fcnn, y_primer_val)
    preds_a = (w_a * pt_cnn + (1 - w_a) * pt_fcnn).argmax(axis=1)
    res_a = metrics_triple(y_primer_test, preds_a)
    res_a['best_cnn_tl_weight'] = w_a
    res_a['tune_on'] = 'primer_val'
    print(f'  [A Primer-val tune] w={w_a:.2f}  val_macroF1={val_f1_a:.4f} → test Macro={res_a["macro_f1"]:.4f}  Acc={res_a["accuracy"]:.4f}')

    # ── Strategi B: tune w on source val ──
    s_img, s_lm, s_y = load_source_val(dataset_name, num_classes)
    s_cnn_loader = make_cnn_loader(s_img, s_y)
    s_fcnn_loader = make_fcnn_loader(s_lm, s_y)
    s_cnn = batched_softmax(cnn_tl, s_cnn_loader)
    s_fcnn = batched_softmax(fcnn, s_fcnn_loader)
    w_b, val_f1_b = grid_search_w(s_cnn, s_fcnn, s_y)
    preds_b = (w_b * pt_cnn + (1 - w_b) * pt_fcnn).argmax(axis=1)
    res_b = metrics_triple(y_primer_test, preds_b)
    res_b['best_cnn_tl_weight'] = w_b
    res_b['tune_on'] = 'source_val'
    print(f'  [B Source-val tune] w={w_b:.2f}  src_val_macroF1={val_f1_b:.4f} → test Macro={res_b["macro_f1"]:.4f}  Acc={res_b["accuracy"]:.4f}')

    # Save to per-source JSON + combined
    cross_file = CROSS_DIR / f'cross_{dataset_name}_{num_classes}c.json'
    existing = {}
    if cross_file.exists():
        with open(cross_file) as f:
            existing = json.load(f)
    existing['Late_Fusion_TL_B1'] = res_a          # default = strategi A
    existing['Late_Fusion_TL_B1_srcval'] = res_b   # strategi B sebagai variant
    with open(cross_file, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'  Updated: {cross_file.name}')

    return {'A': res_a, 'B': res_b}


print('Helpers ready.')

## Run Cross-Dataset Late Fusion TL (4 datasets × 2 classes)

In [ ]:
all_results = {}

# CK+
all_results['ckplus_7c'] = cross_late_fusion_tl('ckplus', 7)
all_results['ckplus_4c'] = cross_late_fusion_tl('ckplus', 4)

# JAFFE
all_results['jaffe_7c'] = cross_late_fusion_tl('jaffe', 7)
all_results['jaffe_4c'] = cross_late_fusion_tl('jaffe', 4)

# RAF-DB
all_results['rafdb_7c'] = cross_late_fusion_tl('rafdb', 7)
all_results['rafdb_4c'] = cross_late_fusion_tl('rafdb', 4)

# KDEF
all_results['kdef_7c'] = cross_late_fusion_tl('kdef', 7)
all_results['kdef_4c'] = cross_late_fusion_tl('kdef', 4)

In [ ]:
# ── Update combined all_cross_results.json ──

combined_file = CROSS_DIR / 'all_cross_results.json'
if combined_file.exists():
    with open(combined_file) as f:
        combined = json.load(f)
else:
    combined = {}

for src_key, strategies in all_results.items():
    if strategies is None:
        continue
    if src_key not in combined:
        combined[src_key] = {}
    combined[src_key]['Late_Fusion_TL_B1'] = strategies['A']
    combined[src_key]['Late_Fusion_TL_B1_srcval'] = strategies['B']

with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2)
print(f'Updated combined: {combined_file.name}')

## Ringkasan — Strategi A vs B

In [ ]:
print(f"\n{'='*92}")
print(f'  Cross-Dataset Late Fusion TL -> Primer test (929 imgs)')
print(f'  Strategi A: w tuned on Primer val | Strategi B: w tuned on source val')
print(f"{'='*92}")
print(f"  {'Source':<14} {'Strategy':<6} {'w':>5} {'Macro':>8} {'Micro':>8} {'Weighted':>10} {'Acc':>8}")
print(f"  {'-'*80}")
for src_key, strategies in all_results.items():
    if strategies is None:
        continue
    for label, res in [('A (pri)', strategies['A']), ('B (src)', strategies['B'])]:
        print(f"  {src_key:<14} {label:<6} {res['best_cnn_tl_weight']:>5.2f} "
              f"{res['macro_f1']:>8.4f} {res['micro_f1']:>8.4f} "
              f"{res['weighted_f1']:>10.4f} {res['accuracy']:>8.4f}")

# Gap A vs B
print(f"\nGap analysis (Strategi A - B, positive = target tuning helps):")
print(f"  {'Source':<14} {'Δ Macro F1':>12} {'w_A':>6} {'w_B':>6}")
for src_key, strategies in all_results.items():
    if strategies is None:
        continue
    gap = strategies['A']['macro_f1'] - strategies['B']['macro_f1']
    print(f"  {src_key:<14} {gap:+12.4f} {strategies['A']['best_cnn_tl_weight']:>6.2f} {strategies['B']['best_cnn_tl_weight']:>6.2f}")